# 00 · Reproduce the proposal's December checkpoint

The proposal presentation showed three 28-day fixed-origin baselines. This notebook reproduces that exact check from the audited current M5 panel. It is a historical checkpoint with dates and formulas fixed by the submission. Start the current workflow at Notebook 01, then return here after ingestion if you want to inspect the earlier benchmark.

Historical scope: CA_1, training January 1, 2014 to December 3, 2015, one origin on December 3, then December 4–31 validation. All 28 predictions for an item use history observable at that single origin. The new experiment updates forecasts daily, uses a different fit period, and assesses a different window. This notebook provides context and does not contribute to the current model selection or final metrics.

Sources: submitted proposal presentation, slide titled “Forecast accuracy”; submitted progress report, Table 7; submitted notebook 2 in the [course submission archive](../references/course_submissions/).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((path for path in (cwd, *cwd.parents) if (path / "configs/config.yaml").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_audit import load_config
from src.features import OriginStore

config = load_config(root / "configs/config.yaml")
processed = root / config["data"]["processed_dir"]
if not (processed / "panel/CA_1.parquet").is_file():
    raise FileNotFoundError("Run Notebook 01 with CA_1 in the store list before reproducing this checkpoint.")

store = OriginStore.from_processed(processed, "CA_1", config)
origin = pd.Timestamp("2015-12-03")
training_dates = pd.date_range("2014-01-01", origin)
training_indexes = store.days.get_indexer(training_dates)
assert (training_indexes >= 0).all()
history = store.sales[:, training_indexes].astype(np.float64)
assert history.shape == (3049, 702)
first = store.origins([origin])
historical_settings = store.options["baselines"]
assert historical_settings["average_window_days"] == 28 and historical_settings["seasonal_period_days"] == 7

## Why hold the origin fixed?

Last observation repeats the last sale 28 times. Seasonal naïve repeats the last seven observed days four times. Moving average repeats the latest 28-day mean. Holding origin fixed answers a different question from the revised daily-review experiment, in which each review can use additional completed days.

RMSSE here divides each item's RMSE by a scale based on **all** first differences in historical training, including leading zeros. The reported RMSE is the mean of per-item RMSE values. The current notebook reports pooled RMSE and uses training scales that start after the first positive sale.

In [ ]:
validation = {"start": "2015-12-04", "end": "2015-12-31"}
methods = {"Last observation": "naive", "Seasonal naïve": "seasonal_naive", "28-day moving average": "moving_average"}
predictions = [store.design(first, horizon, family="daily", labels=validation).set_index("item_id")
               for horizon in range(1, 29)]
for day in predictions:
    assert day.index.tolist() == store.items.tolist()
actual = np.column_stack([day["actual"] for day in predictions])
assert actual.shape == (3049, 28)
scale = np.mean(np.diff(history, axis=1) ** 2, axis=1)
scale[scale == 0] = np.nan
scores = []
for label, name in methods.items():
    path = np.column_stack([day[name] for day in predictions]).astype(float)
    error = path - actual
    mse = np.mean(error ** 2, axis=1)
    scores.append(pd.DataFrame({
        "SKU": store.items, "method": label, "MAE": np.mean(np.abs(error), axis=1),
        "RMSE": np.sqrt(mse), "RMSSE": np.sqrt(mse / scale),
    }))
scores = pd.concat(scores, ignore_index=True)
reproduced = scores.groupby("method", sort=False).agg(MAE=("MAE", "mean"), RMSE=("RMSE", "mean"),
                                                       RMSSE=("RMSSE", "mean"), scalable_items=("RMSSE", "count"))
published = pd.DataFrame(
    [[1.2393, 1.6764, .8950], [1.2335, 1.7425, .9259], [1.0887, 1.4016, .7234]],
    index=list(methods), columns=["MAE", "RMSE", "RMSSE"],
)
display(reproduced.round(4))
pd.testing.assert_frame_equal(reproduced[["MAE", "RMSE", "RMSSE"]].round(4), published.rename_axis("method"))
print("PASS: all nine published baseline scores reproduced to four decimals.")

## Did the mean hide differences across categories?

Keep the original category-level check. Categories describe the merchandise hierarchy and were available before the project defined quantitative demand regimes. They answer a different question from stable, volatile or intermittent sales patterns.

In [ ]:
mapping = pd.read_parquet(processed / "panel/CA_1.parquet", columns=["item_id", "cat_id", "date"],
                          filters=[("date", "==", origin)]).rename(columns={"item_id": "SKU"})
scores = scores.merge(mapping[["SKU", "cat_id"]], on="SKU", validate="many_to_one")
display(scores.groupby(["cat_id", "method"], observed=True)[["MAE", "RMSE", "RMSSE"]].mean().round(4))
display(scores.groupby("method")[["MAE", "RMSSE"]].quantile([.25, .5, .75, .9]).round(4))
fig, ax = plt.subplots(figsize=(8.4, 3.5), layout="constrained")
series_mae = scores.groupby("method", sort=False)["MAE"]
ax.boxplot([group.to_numpy() for _, group in series_mae], tick_labels=list(methods), showfliers=False)
ax.set(title="December baseline errors by SKU", ylabel="MAE (units)")
plt.show()

The moving average leads all three aggregate error measures in this historical window. That is why it deserves a central role as a comparator. The progress report later repeated the comparison on different dates using daily updates and compared outcomes in a simplified inventory simulation. See the [stage comparison](../docs/PROJECT_EVOLUTION.md) for how those results connect to the current project.